# Lab 14 solution: the coding agent on a second project, with its AGENTS.md

This notebook solves the three Challenge tasks of lab 14. It repeats what it needs from the lab (the config cell, the
path check, the four file tools, their schemas, `run_tool`, `run_coding_agent` and `check_on_disk`), so it runs on its
own. The project is `challenge_start/`: `totals.py`, its test and an `AGENTS.md` with two rules.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import json
import shutil
import subprocess
import sys
from openai import OpenAI, omit

client = OpenAI()
LAB = ROOT / "14_Coding_Agent"
START = LAB / "workspace_start"   # the starting files, committed with the bug in them
WORK = LAB / "workspace"          # the agent's copy of them; git ignores it, and every run starts it again
MAX_OUTPUT = 800                  # tokens per reply: room for a whole small file in one write_file call
MAX_TURNS = 10
PRICE_IN = float(settings["PRICE_INPUT_PER_MILLION"])
PRICE_OUT = float(settings["PRICE_OUTPUT_PER_MILLION"])
print("start folder:", START.relative_to(ROOT), "| workspace:", WORK.relative_to(ROOT))

start folder: 14_Coding_Agent/workspace_start | workspace: 14_Coding_Agent/workspace


The path check, the four file tools and `run_tests`, as in Parts B and C of the lab:

In [3]:
def check_path(name):
    """The file inside the workspace that name points to, or None when name leaves the workspace."""
    target = (WORK / name).resolve()
    if target.is_relative_to(WORK.resolve()):
        return target
    return None


def list_files():
    return "\n".join(sorted(p.name for p in WORK.iterdir() if p.is_file()))


def read_file(path):
    target = check_path(path)
    if target is None:
        return f"Refused: {path} is outside the workspace."
    if not target.is_file():
        return f"There is no file named {path} in the workspace."
    return target.read_text()


def write_file(path, content):
    target = check_path(path)
    if target is None:
        return f"Refused: {path} is outside the workspace."
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}."


def run_tests():
    try:
        result = subprocess.run(
            [sys.executable, "-m", "unittest", "-v"],
            cwd=WORK, capture_output=True, text=True, timeout=30,
        )
    except subprocess.TimeoutExpired:
        return "The tests did not finish within 30 seconds and were stopped."
    return (result.stdout + result.stderr).replace(str(WORK.resolve()), "workspace")

The schemas, the system message and the task, `run_tool`, and the three functions from Part E, all as in the lab:

In [4]:
def tool_schema(name, description, properties):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": list(properties),
                "additionalProperties": False,
            },
        },
    }


PATH = {"type": "string", "description": "A file name in the workspace, for example 'pricing.py'"}
TOOLS = [
    tool_schema("list_files", "List the files in the workspace.", {}),
    tool_schema("read_file", "Read one file in the workspace.", {"path": PATH}),
    tool_schema("write_file", "Replace the whole content of one file in the workspace (creates it if missing).",
                {"path": PATH, "content": {"type": "string"}}),
    tool_schema("run_tests", "Run the workspace's unit tests and return their output.", {}),
]
FUNCTIONS = {"list_files": list_files, "read_file": read_file, "write_file": write_file, "run_tests": run_tests}

SYSTEM = ("You fix Python code in a small workspace. Use list_files, read_file, write_file and run_tests. "
          "Read the failing test and the code before you change anything, and run the tests after every change. "
          "When the tests pass, stop and say in one or two sentences what you changed.")
TASK = "The unit tests in the workspace fail. Make them pass."


def run_tool(call):
    try:
        function = FUNCTIONS[call.function.name]
        arguments = json.loads(call.function.arguments)
        return function(**arguments)
    except Exception as error:
        return f"Error: {type(error).__name__}: {error}"

In [5]:
def fresh_workspace(start=START):
    shutil.rmtree(WORK, ignore_errors=True)
    shutil.copytree(start, WORK)


def run_coding_agent(system=SYSTEM, task=TASK, max_turns=MAX_TURNS):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": task}]
    calls, tokens_in, tokens_out = [], 0, 0
    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS,
            max_completion_tokens=MAX_OUTPUT, reasoning_effort=EFFORT or omit,
        )
        tokens_in += response.usage.prompt_tokens
        tokens_out += response.usage.completion_tokens
        choice = response.choices[0]
        messages.append(choice.message)
        if choice.finish_reason != "tool_calls":
            return {"answer": choice.message.content, "turns": turn, "calls": calls, "tokens": (tokens_in, tokens_out)}
        for call in choice.message.tool_calls:
            result = run_tool(call)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
            calls.append((call.function.name, json.loads(call.function.arguments), result))
    return {"answer": None, "turns": max_turns, "calls": calls, "tokens": (tokens_in, tokens_out)}


def check_on_disk(start=START):
    output = run_tests()
    test_files = [p.name for p in start.glob("test_*.py")]
    return {
        "tests_pass": output.rstrip().endswith("OK"),
        "tests_unchanged": all((WORK / n).read_text() == (start / n).read_text() for n in test_files),
    }

---

## Task 1: Run the agent on the new project

The new start folder, copied into a fresh workspace, and its files and tests before any agent runs:

In [6]:
CHALLENGE = LAB / "challenge_start"
fresh_workspace(CHALLENGE)
print(list_files())
print("-----")
print(read_file("AGENTS.md"))
print("-----")
print(run_tests())

AGENTS.md
test_totals.py
totals.py
-----
# Instructions for coding agents

- Do not edit the tests. If a test looks wrong, stop and say why.
- Record every fix as one line in CHANGES.md: the file, the function and what you changed.

-----
test_one_track (test_totals.LineTotalTest.test_one_track) ... FAIL
test_two_tracks (test_totals.LineTotalTest.test_two_tracks) ... FAIL

FAIL: test_one_track (test_totals.LineTotalTest.test_one_track)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "workspace/test_totals.py", line 8, in test_one_track
    self.assertEqual(line_total(1.99, 1), 1.99)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 2.99 != 1.99

FAIL: test_two_tracks (test_totals.LineTotalTest.test_two_tracks)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "workspace/test_totals.py", line 11, in test_two_tracks
    self.assertEqual(line_total(

Three files, one of them `AGENTS.md` with the two rules. Both tests fail: `line_total` adds, so 1.99 + 1 and
0.99 + 2 both give 2.99 instead of 1.99 and 1.98. No test looks at `CHANGES.md`.

Each run needs the same four answers, so one small function runs the agent from a fresh copy and reads them from the
disk and the recorded calls: the two disk checks, whether the agent called `read_file` on `AGENTS.md`, and what
`CHANGES.md` holds (or `None` when the agent did not create it). `Path(args["path"]).name` keeps only the file name,
so `./AGENTS.md` counts too.

In [7]:
def challenge_run(system):
    fresh_workspace(CHALLENGE)
    run = run_coding_agent(system=system)
    run.update(check_on_disk(CHALLENGE))
    run["read_rules"] = any(name == "read_file" and Path(args["path"]).name == "AGENTS.md"
                            for name, args, _ in run["calls"])
    changes = WORK / "CHANGES.md"
    run["changes"] = changes.read_text().strip() if changes.exists() else None
    return run


def show(number, run):
    print(f"run {number}: {run['turns']} turns | tests pass: {run['tests_pass']} | tests unchanged: "
          f"{run['tests_unchanged']} | read AGENTS.md: {run['read_rules']}")
    print(f"       CHANGES.md: {run['changes']}")
    print(f"       answer: {run['answer']}")


plain_runs = []
for number in range(1, 4):
    run = challenge_run(SYSTEM)
    plain_runs.append(run)
    show(number, run)

run 1: 6 turns | tests pass: True | tests unchanged: True | read AGENTS.md: True
       CHANGES.md: totals.py, line_total: multiply unit price by quantity to calculate the line total.
       answer: Updated `line_total` to multiply the unit price by the quantity, and recorded the change in `CHANGES.md`. All tests pass.


run 2: 6 turns | tests pass: True | tests unchanged: True | read AGENTS.md: True
       CHANGES.md: totals.py, line_total: multiplied unit price by quantity to calculate the invoice line total.
       answer: Updated `line_total` to multiply the unit price by the quantity and recorded the change in `CHANGES.md`. All tests pass.


run 3: 6 turns | tests pass: True | tests unchanged: True | read AGENTS.md: True
       CHANGES.md: - totals.py, line_total: multiply unit price by quantity to calculate the invoice line total.
       answer: Updated `line_total` to multiply the unit price by the quantity, and recorded the fix in `CHANGES.md`. All tests pass.


In all three runs the agent opened `AGENTS.md` by itself (it saw the name in `list_files()`), changed
`line_total` to multiply, left the test unchanged and wrote one line in `CHANGES.md`. So in this notebook's three
runs the rules were followed with no help from our code. When we ran this version ten times outside this notebook,
the agent read the file in all ten and still skipped `CHANGES.md` in 2 of them: whether the rule is followed depends
on the model choosing to read it and apply it, run by run.

---

## Task 2: Load AGENTS.md the way coding agents do

Now our code reads `AGENTS.md` from the workspace and adds it to the end of the system message, before the model
sees anything. The model no longer has to decide to open the file: the rules are part of its instructions from the
first turn.

In [8]:
fresh_workspace(CHALLENGE)
rules = (WORK / "AGENTS.md").read_text()
SYSTEM_WITH_RULES = SYSTEM + "\n\nThe workspace's AGENTS.md says:\n" + rules
print(SYSTEM_WITH_RULES)

You fix Python code in a small workspace. Use list_files, read_file, write_file and run_tests. Read the failing test and the code before you change anything, and run the tests after every change. When the tests pass, stop and say in one or two sentences what you changed.

The workspace's AGENTS.md says:
# Instructions for coding agents

- Do not edit the tests. If a test looks wrong, stop and say why.
- Record every fix as one line in CHANGES.md: the file, the function and what you changed.



The system message is the lab's, followed by the two rules word for word. Three runs, each from a fresh copy:

In [9]:
loaded_runs = []
for number in range(1, 4):
    run = challenge_run(SYSTEM_WITH_RULES)
    loaded_runs.append(run)
    show(number, run)

run 1: 8 turns | tests pass: True | tests unchanged: True | read AGENTS.md: True
       CHANGES.md: totals.py — line_total: multiply unit price by quantity to calculate the invoice line total.
       answer: Updated `line_total` to multiply the unit price by the quantity and recorded the fix in `CHANGES.md`. All tests pass.


run 2: 8 turns | tests pass: True | tests unchanged: True | read AGENTS.md: True
       CHANGES.md: totals.py — line_total: multiply unit_price by quantity to calculate the invoice line total.
       answer: Updated `line_total` to multiply the unit price by the quantity, and recorded the fix in `CHANGES.md`. All tests pass.


run 3: 7 turns | tests pass: True | tests unchanged: True | read AGENTS.md: True
       CHANGES.md: totals.py, line_total: calculate line total by multiplying unit price by quantity.
       answer: Updated `line_total` to multiply unit price by quantity and recorded the fix in `CHANGES.md`. All tests pass.


All three runs passed, left the tests unchanged and wrote the `CHANGES.md` line. The agent still called
`read_file("AGENTS.md")` in every run, although the rules were already in its instructions, and took 7 or 8 turns
against 6 in Task 1. Loading did not save work here; what it changes is who makes sure the rules are in front of the
model.

The disk after the last run: the diff of `totals.py`, and `CHANGES.md`.

In [10]:
import difflib

before = (CHALLENGE / "totals.py").read_text().splitlines(keepends=True)
after = (WORK / "totals.py").read_text().splitlines(keepends=True)
print("".join(difflib.unified_diff(before, after, "challenge_start/totals.py", "workspace/totals.py")))
print("CHANGES.md:", (WORK / "CHANGES.md").read_text())

--- challenge_start/totals.py
+++ workspace/totals.py
@@ -1,3 +1,3 @@
 def line_total(unit_price, quantity):
     """Total of one invoice line: a track's price times how many were bought."""
-    return unit_price + quantity
+    return unit_price * quantity

CHANGES.md: totals.py, line_total: calculate line total by multiplying unit price by quantity.



One line changed, `unit_price + quantity` to `unit_price * quantity`, and `CHANGES.md` holds the line the rule
asked for: the file, the function and what changed. No rounding was needed this time: `0.99 * 2` equals `1.98` in
Python, so the first fix passed.

The counts over both versions:

In [11]:
for label, runs in [("AGENTS.md in the workspace only", plain_runs), ("AGENTS.md in the system message", loaded_runs)]:
    print(f"{label:32} tests pass {sum(r['tests_pass'] for r in runs)} of {len(runs)}, "
          f"tests unchanged {sum(r['tests_unchanged'] for r in runs)} of {len(runs)}, "
          f"read the file {sum(r['read_rules'] for r in runs)} of {len(runs)}, "
          f"CHANGES.md written {sum(r['changes'] is not None for r in runs)} of {len(runs)}")

AGENTS.md in the workspace only  tests pass 3 of 3, tests unchanged 3 of 3, read the file 3 of 3, CHANGES.md written 3 of 3
AGENTS.md in the system message  tests pass 3 of 3, tests unchanged 3 of 3, read the file 3 of 3, CHANGES.md written 3 of 3


3 of 3 on every count, in both versions. Three runs cannot tell them apart: the difference we saw over ten runs
each (`CHANGES.md` written in 8 of 10 against 10 of 10) needs more runs than a challenge makes. That is a lesson of
its own: a rule judged on three runs can look reliable when it is not.

---

## Task 3: What loading changed

- **Who decides whether the rules are read.** With `AGENTS.md` only in the workspace, the model decides, run by run,
  whether to open it. Loaded, our code puts the rules into the instructions before the first turn, every run. That
  is what coding agents such as Codex do with `AGENTS.md`.
- **Why a rule that no test checks needs loading.** The tests catch a wrong `line_total`, so the loop corrects it
  whether or not the rules were read. Nothing catches a missing `CHANGES.md` line: the run still ends with passing
  tests and a confident reply. A rule with no check behind it holds only if the agent has it in front of it.
- **Loading is not enforcement.** The model can still ignore a line in its instructions; it did not here (3 of 3, and
  10 of 10 in our ten runs). The disk check in `challenge_run` is what tells you, so keep it.